# 14 · Stage 3 v5-B — V-JEPA 2.1 last-2-block fine-tuning

v5-A best checkpoint를 그대로 warm-start하고 V-JEPA 2.1 ViT-B의 마지막 두
transformer block만 작은 learning rate로 푼다.

- block 0..9: frozen + eval
- block 10: train, LR 2e-6
- block 11 + final hierarchical tap norm: train, LR 4e-6
- v5-A head/spatial/auxiliary branches: train, LR 1.5e-5
- T=32 / A2D2 / event-balanced sampler / v5 loss는 그대로 유지
- validation은 전체 12k windows의 앞 800개가 아니라, 전체 구간에서
  deterministic하게 균등 선택한 800개를 사용

첫 smoke에서 frozen block에 gradient가 생기지 않고 blocks 10/11에는 실제
gradient가 생기는지, 그리고 L4 peak memory를 확인한 뒤 학습한다.


In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

try:
    drive.mount("/content/drive", force_remount=False)
except Exception:
    drive.mount("/content/drive", force_remount=True)

REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not (REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         "--single-branch", REPO_URL, str(REPO)],
        check=True,
    )
else:
    current = subprocess.run(
        ["git", "-C", str(REPO), "branch", "--show-current"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if current != BRANCH:
        subprocess.run(
            ["git", "-C", str(REPO), "checkout", BRANCH],
            check=True,
        )
    dirty = subprocess.run(
        ["git", "-C", str(REPO), "status", "--porcelain"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if not dirty:
        subprocess.run(
            ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
            check=True,
        )
    else:
        print("WARNING: local repo dirty; git pull skipped")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        "timm==1.0.15",
        "fvcore==0.1.5.post20221221",
        "iopath==0.1.10",
        "yacs==0.1.8",
        "einops==0.8.1",
        "wandb==0.29.0",
        "easydict==1.13",
    ],
    check=True,
)

SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import torch
import yaml

from blackbox_detection.stage3.proxy_metrics import assert_dacon_metric_contract
from blackbox_detection.utils import (
    dataloader_seed_kwargs,
    finish_wandb,
    init_wandb,
    seed_everything,
)

DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"
COMMA_ROOT = DATA_ROOT / "comma2k19" / "processed" / "v1"
A2D2_ROOT = DATA_ROOT / "A2D2" / "processed"
MANIFEST_ROOT = DRIVE_ROOT / "manifests/stage3/v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs/stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"
WANDB_KEY_PATH = DRIVE_ROOT / "wandb_key.txt"

LOCAL_PRETRAINED_ROOT = Path("/content/pretrained")
LOCAL_OUTPUT_ROOT = Path("/content/stage3_runs")
for p in [OUTPUT_ROOT, PRETRAINED_ROOT, LOCAL_PRETRAINED_ROOT, LOCAL_OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

CFG_PATH = REPO / "configs/stage3/vjepa21b_can_v5b.yaml"
cfg = yaml.safe_load(CFG_PATH.read_text(encoding="utf-8"))
stats = json.loads(
    (MANIFEST_ROOT / "target_stats.json").read_text(encoding="utf-8")
)

SEED = int(cfg["seed"])
seed_everything(SEED, deterministic=False)
assert_dacon_metric_contract()

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

loss_cfg = copy.deepcopy(cfg["loss"])
loss_cfg["base"]["normalization"] = {
    name: {
        "mean": float(stats[name]["mean"]),
        "std": float(stats[name]["std"]),
    }
    for name in ("speed_mps", "accel_from_speed_mps2")
}

print("Python         :", sys.version)
print("Torch          :", torch.__version__)
print("GPU            :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("Git            :", BRANCH, GIT_COMMIT)
print("Config         :", CFG_PATH)
print("metric contract: PASS")


In [ ]:
def _is_usable_file(path: Path, min_bytes: int = 1) -> bool:
    try:
        return path.is_file() and path.stat().st_size >= int(min_bytes)
    except OSError:
        return False

def copy_file_to_local(
    source: Path,
    destination: Path,
    *,
    min_bytes: int = 1,
    retries: int = 2,
) -> bool:
    destination.parent.mkdir(parents=True, exist_ok=True)
    last_error = None
    for attempt in range(1, retries + 2):
        tmp = destination.with_name(destination.name + ".copy.tmp")
        try:
            tmp.unlink(missing_ok=True)
            with source.open("rb") as src, tmp.open("wb") as dst:
                shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
            if tmp.stat().st_size < int(min_bytes):
                raise OSError(f"staged file too small: {tmp.stat().st_size}")
            os.replace(tmp, destination)
            return True
        except OSError as exc:
            last_error = exc
            tmp.unlink(missing_ok=True)
            print(f"copy attempt {attempt} failed:", repr(exc))
            if attempt <= retries:
                time.sleep(2 * attempt)
    print("copy failed:", repr(last_error))
    return False

VJEPA_REPO = Path("/content/vjepa2")
VJEPA_COMMIT = "45d025f636dfc58fc2426905fc4a1ab755b1c3e5"

if not (VJEPA_REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "-q",
         "https://github.com/facebookresearch/vjepa2.git",
         str(VJEPA_REPO)],
        check=True,
    )
subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "fetch", "--all", "--tags"],
    check=True,
)
subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "checkout", "-q", VJEPA_COMMIT],
    check=True,
)

ACTUAL_VJEPA_COMMIT = subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

CKPT_NAME = "vjepa2_1_vitb_dist_vitG_384.pt"
VJEPA_CKPT_DRIVE = PRETRAINED_ROOT / CKPT_NAME
VJEPA_CKPT = LOCAL_PRETRAINED_ROOT / CKPT_NAME
VJEPA_CKPT_URL = (
    "https://dl.fbaipublicfiles.com/vjepa2/"
    "vjepa2_1_vitb_dist_vitG_384.pt"
)
MIN_VJEPA_BYTES = 1_000_000_000

if not _is_usable_file(VJEPA_CKPT, MIN_VJEPA_BYTES):
    copied = copy_file_to_local(
        VJEPA_CKPT_DRIVE, VJEPA_CKPT,
        min_bytes=MIN_VJEPA_BYTES,
    )
    if not copied:
        tmp = VJEPA_CKPT.with_name(VJEPA_CKPT.name + ".download.tmp")
        tmp.unlink(missing_ok=True)
        subprocess.run(
            ["wget", "-q", "--show-progress", "-O", str(tmp), VJEPA_CKPT_URL],
            check=True,
        )
        if tmp.stat().st_size < MIN_VJEPA_BYTES:
            raise RuntimeError("V-JEPA checkpoint download is too small")
        os.replace(tmp, VJEPA_CKPT)

print("V-JEPA commit:", ACTUAL_VJEPA_COMMIT)
print("V-JEPA ckpt  :", VJEPA_CKPT)


## Dataset / cached event sampler / unbiased 800-window validation


In [ ]:
from torch.utils.data import DataLoader, Subset

from blackbox_detection.stage3.v5_dataset import (
    MixedStage3CANDataset,
    build_event_balanced_sampler,
)

dc = cfg["data"]
tc = cfg["training"]
source_cfg = dc["sources"]

a2d2_manifest = pd.read_csv(A2D2_ROOT / "manifest.csv")
assert int(a2d2_manifest["num_frames"].sum()) == 9151

train_sources = [
    {
        "name": "comma2k19",
        "manifest": MANIFEST_ROOT / "comma_train.csv",
        "processed_root": COMMA_ROOT,
        **source_cfg["comma2k19"],
    },
    {
        "name": "a2d2",
        "manifest": a2d2_manifest,
        "processed_root": A2D2_ROOT,
        **source_cfg["a2d2"],
    },
]
val_sources = [
    {
        "name": "comma2k19",
        "manifest": MANIFEST_ROOT / "comma_val_id.csv",
        "processed_root": COMMA_ROOT,
        **source_cfg["comma2k19"],
    },
]

train_ds = MixedStage3CANDataset(
    train_sources,
    target_stats=stats,
    clip_len=dc["clip_len"],
    window_stride=dc["train_window_stride"],
    input_size=(dc["input_height"], dc["input_width"]),
    random_flip=dc["train_random_flip"],
    seed=SEED,
)
val_ds = MixedStage3CANDataset(
    val_sources,
    target_stats=stats,
    clip_len=dc["clip_len"],
    window_stride=dc["val_window_stride"],
    input_size=(dc["input_height"], dc["input_width"]),
    random_flip=False,
    seed=SEED + 1,
)

sampler_cfg = dc["sampler"]
EVENT_CACHE = (
    DRIVE_ROOT
    / "manifests/stage3/v5a"
    / f"event_index_t{dc['clip_len']}_s{dc['train_window_stride']}.npz"
)

train_sampler, sampler_report = build_event_balanced_sampler(
    train_ds,
    num_samples=int(tc["max_steps_per_epoch"]) * int(tc["batch_size"]),
    event_multipliers=sampler_cfg["event_multipliers"],
    source_multipliers=sampler_cfg["source_multipliers"],
    inverse_frequency_power=sampler_cfg["inverse_frequency_power"],
    max_normalized_weight=sampler_cfg["max_normalized_weight"],
    seed=SEED,
    cache_path=EVENT_CACHE,
)

# Uniform coverage across the whole validation universe.
n_val_eval = min(int(tc["max_val_steps"]), len(val_ds))
val_indices = np.linspace(
    0, len(val_ds) - 1,
    num=n_val_eval,
    dtype=np.int64,
)
val_indices = np.unique(val_indices)
val_eval_ds = Subset(val_ds, val_indices.tolist())

train_loader = DataLoader(
    train_ds,
    batch_size=tc["batch_size"],
    sampler=train_sampler,
    shuffle=False,
    num_workers=dc["num_workers"],
    pin_memory=True,
    persistent_workers=dc["num_workers"] > 0,
    **dataloader_seed_kwargs(SEED),
)
val_loader = DataLoader(
    val_eval_ds,
    batch_size=tc["batch_size"],
    shuffle=False,
    num_workers=dc["num_workers"],
    pin_memory=True,
    persistent_workers=dc["num_workers"] > 0,
    **dataloader_seed_kwargs(SEED + 1),
)

print("event cache           :", EVENT_CACHE)
print("expected A2D2 share   :", sampler_report["expected_source_share"]["a2d2"])
print("train windows universe:", len(train_ds))
print("sampled train / epoch :", len(train_sampler))
print("val windows universe  :", len(val_ds))
print("val windows evaluated :", len(val_eval_ds))
assert len(val_eval_ds) == int(tc["max_val_steps"])


## V5-A warm start + last two V-JEPA blocks


In [ ]:
from blackbox_detection.stage3.vjepa21 import load_vjepa21_base_encoder
from blackbox_detection.stage3.v5b_finetune import (
    VJEPA21DenseCANV5B,
    split_v5b_optimizer_parameters,
)
from blackbox_detection.stage3.trainer import build_scheduler
from blackbox_detection.stage3.v5_trainer import V5CANTrainer
from blackbox_detection.utils.checkpoint import load_checkpoint

mc = cfg["model"]
fusion_cfg = dict(mc["accel_fusion"])
spatial_cfg = dict(mc["spatial_pool"])

# Build the exact T=32 V-JEPA architecture.
backbone = load_vjepa21_base_encoder(
    VJEPA_REPO,
    VJEPA_CKPT,
    num_frames=dc["clip_len"],
    out_layers=tuple(mc["out_layers"]),
    freeze=False,
)

model = VJEPA21DenseCANV5B(
    backbone,
    feature_dim=mc["feature_dim"],
    temporal_hidden=mc["temporal_hidden"],
    temporal_layers=mc["temporal_layers"],
    spatial_grid=tuple(spatial_cfg["grid"]),
    spatial_gate_init=spatial_cfg["gate_init"],
    accel_ordinal_thresholds_mps2=mc["accel_ordinal_thresholds_mps2"],
    accel_fusion_enabled=fusion_cfg["enabled"],
    accel_fusion_hidden=fusion_cfg["hidden"],
    accel_fusion_gate_init=fusion_cfg["gate_init"],
    accel_fusion_detach_ordinal_inputs=fusion_cfg["detach_ordinal_inputs"],
    stop_thresholds_mps=mc["stop_thresholds_mps"],
    turn_yaw_thresholds_rps=mc["turn_yaw_thresholds_rps"],
    steer_activity_thresholds=mc["steer_activity_thresholds"],
    brake_thresholds_bar=mc["brake_thresholds_bar"],
    throttle_thresholds_pct=mc["throttle_thresholds_pct"],
)

RUN_VARIANT = cfg["experiment"]["name"]
RUN_DIR = OUTPUT_ROOT / RUN_VARIANT
LOCAL_RUN_DIR = LOCAL_OUTPUT_ROOT / RUN_VARIANT
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

# Resume V5-B if its own checkpoint exists.
for filename in ("latest.pt", "best.pt"):
    persistent = RUN_DIR / filename
    local = LOCAL_RUN_DIR / filename
    if not local.is_file() and _is_usable_file(persistent):
        print(
            "stage own", filename,
            copy_file_to_local(persistent, local),
        )

resume_path = LOCAL_RUN_DIR / "latest.pt"
warm_start_metadata = None

# Fresh V5-B: full strict model warm-start from V5-A best.
if not resume_path.is_file():
    wc = cfg["warm_start"]
    v5a_drive = OUTPUT_ROOT / wc["run_name"] / wc["checkpoint"]
    v5a_local = (
        LOCAL_PRETRAINED_ROOT
        / f"{wc['run_name']}__{wc['checkpoint']}"
    )

    if not _is_usable_file(v5a_local, 1_000_000):
        if not _is_usable_file(v5a_drive, 1_000_000):
            raise FileNotFoundError(v5a_drive)
        if not copy_file_to_local(
            v5a_drive, v5a_local, min_bytes=1_000_000
        ):
            raise OSError("failed to stage v5-A best checkpoint")

    warm_start_metadata = load_checkpoint(
        v5a_local,
        model=model,
        optimizer=None,
        scheduler=None,
        map_location="cpu",
        strict=bool(wc.get("strict", True)),
        restore_rng_state=False,
    )
    assert not warm_start_metadata["missing_keys"]
    assert not warm_start_metadata["unexpected_keys"]
    print("V5-A full strict warm-start: PASS")
    print("source epoch:", warm_start_metadata.get("epoch"))
else:
    print("V5-B latest.pt found: V5-A one-time warm-start skipped")

ft_cfg = mc["backbone_finetune"]
ft_report = model.configure_partial_backbone(
    last_n_blocks=ft_cfg["last_n_blocks"],
    train_final_tap_norm=ft_cfg["train_final_tap_norm"],
)

print("backbone depth              :", ft_report.depth)
print("trainable block indices     :", ft_report.trainable_block_indices)
print("trainable norm indices      :", ft_report.trainable_norm_indices)
print("trainable backbone params M :", ft_report.trainable_backbone_params / 1e6)
print("frozen backbone params M    :", ft_report.frozen_backbone_params / 1e6)

assert ft_report.depth == 12
assert ft_report.trainable_block_indices == (10, 11)


## Discriminative optimizer: head / block10 / block11


In [ ]:
families = split_v5b_optimizer_parameters(model)

def add_decay_groups(
    out,
    *,
    family_name,
    named_params,
    lr,
    weight_decay,
):
    decay = []
    no_decay = []
    for name, p in named_params:
        if p.ndim <= 1 or name.endswith(".bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    if decay:
        out.append({
            "params": decay,
            "lr": float(lr),
            "weight_decay": float(weight_decay),
            "name": f"{family_name}_decay",
        })
    if no_decay:
        out.append({
            "params": no_decay,
            "lr": float(lr),
            "weight_decay": 0.0,
            "name": f"{family_name}_no_decay",
        })

optimizer_groups = []
add_decay_groups(
    optimizer_groups,
    family_name="head",
    named_params=families["head"],
    lr=tc["head_learning_rate"],
    weight_decay=tc["head_weight_decay"],
)
add_decay_groups(
    optimizer_groups,
    family_name="block10",
    named_params=families["backbone_penultimate"],
    lr=tc["penultimate_block_learning_rate"],
    weight_decay=tc["backbone_weight_decay"],
)
add_decay_groups(
    optimizer_groups,
    family_name="block11",
    named_params=families["backbone_last"],
    lr=tc["last_block_learning_rate"],
    weight_decay=tc["backbone_weight_decay"],
)

optimizer = torch.optim.AdamW(optimizer_groups)

micro_steps_per_epoch = min(
    len(train_loader), int(tc["max_steps_per_epoch"])
)
optimizer_steps_per_epoch = max(
    math.ceil(
        micro_steps_per_epoch / int(tc["grad_accum_steps"])
    ),
    1,
)
total_optimizer_steps = (
    optimizer_steps_per_epoch * int(tc["epochs"])
)

scheduler = build_scheduler(
    optimizer,
    total_steps=total_optimizer_steps,
    warmup_ratio=tc["warmup_ratio"],
    min_ratio=tc["min_learning_rate_ratio"],
)

for group in optimizer.param_groups:
    print(
        group["name"],
        "lr=", group["lr"],
        "wd=", group["weight_decay"],
        "params=", sum(p.numel() for p in group["params"]),
    )

print("optimizer steps/epoch:", optimizer_steps_per_epoch)
print("total optimizer steps:", total_optimizer_steps)


## W&B + trainer


In [ ]:
lc = cfg.get("logging", {})
WANDB_ENABLED = bool(lc.get("wandb_enabled", True))
wandb_run = None

if WANDB_ENABLED:
    import wandb

    if not WANDB_KEY_PATH.is_file():
        raise FileNotFoundError(WANDB_KEY_PATH)
    wandb.login(
        key=WANDB_KEY_PATH.read_text(encoding="utf-8").strip(),
        relogin=False,
    )
    finish_wandb()

    run_id_path = RUN_DIR / "wandb_run_id.txt"
    stored_run_id = (
        run_id_path.read_text(encoding="utf-8").strip()
        if run_id_path.is_file()
        else None
    )
    stored_run_id = stored_run_id or None

    wandb_run = init_wandb(
        project=str(lc.get("wandb_project", "blackbox-stage3")),
        entity=os.getenv("WANDB_ENTITY") or None,
        name=f"{RUN_VARIANT}__seed{SEED}__{GIT_COMMIT}",
        group=str(lc.get("wandb_group", "vjepa21b_can_v5b")),
        tags=[
            "stage3", "vjepa2.1", "v5b", "a2d2",
            "t32", "last2-finetune", "event-balanced",
        ],
        run_id=stored_run_id,
        resume="allow",
        config={
            "git_commit": GIT_COMMIT,
            "vjepa_commit": ACTUAL_VJEPA_COMMIT,
            "run_variant": RUN_VARIANT,
            "seed": SEED,
            "finetune_report": ft_report.__dict__,
            "sampler_report": sampler_report,
        },
    )
    if not stored_run_id and wandb_run is not None:
        run_id_path.write_text(str(wandb_run.id), encoding="utf-8")

vc = cfg["validation"]
trainer = V5CANTrainer(
    model,
    optimizer,
    scheduler=scheduler,
    device="cuda" if torch.cuda.is_available() else "cpu",
    grad_accum_steps=tc["grad_accum_steps"],
    grad_clip_norm=tc["grad_clip_norm"],
    amp_dtype=tc["amp_dtype"],
    loss_weights=loss_cfg,
    stats=stats,
    proxy_rules=vc["proxy_rules"],
    output_dir=LOCAL_RUN_DIR,
    sync_dir=RUN_DIR,
    wandb_enabled=WANDB_ENABLED,
    log_interval=tc["log_interval"],
    config=cfg,
)

print("trainer device:", trainer.device)


## Gradient + memory smoke — 여기까지 PASS 후 학습


In [ ]:
from blackbox_detection.stage3.v5_losses import v5_multitask_loss

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

smoke = next(iter(train_loader))
video = smoke["video"][:1].to(trainer.device)
target = smoke["target"][:1].to(trainer.device)
valid = smoke["valid"][:1].to(trainer.device)
aux = {
    k: v[:1].to(trainer.device)
    for k, v in smoke["aux"].items()
}

optimizer.zero_grad(set_to_none=True)
model.train()

# Mode contract: frozen backbone globally eval, selected blocks train.
assert model.backbone.training is False
assert model.backbone.blocks[9].training is False
assert model.backbone.blocks[10].training is True
assert model.backbone.blocks[11].training is True

with torch.autocast(
    device_type=trainer.device.type,
    dtype=trainer.amp_dtype,
    enabled=trainer.device.type == "cuda",
):
    out = model(video)
    smoke_loss, smoke_parts = v5_multitask_loss(
        out,
        target,
        valid,
        aux,
        loss_cfg,
        stats=stats,
    )

assert torch.isfinite(smoke_loss), smoke_loss
smoke_loss.backward()

# Frozen block must stay gradient-free.
assert all(
    p.grad is None
    for p in model.backbone.blocks[9].parameters()
)

def grad_norm(module):
    grads = [
        p.grad.detach().float().norm()
        for p in module.parameters()
        if p.grad is not None
    ]
    if not grads:
        return 0.0
    return float(torch.stack(grads).norm().cpu())

g10 = grad_norm(model.backbone.blocks[10])
g11 = grad_norm(model.backbone.blocks[11])
g_spatial = grad_norm(model.spatial_pool)

assert g10 > 0 and np.isfinite(g10)
assert g11 > 0 and np.isfinite(g11)
assert g_spatial > 0 and np.isfinite(g_spatial)

peak_gib = (
    torch.cuda.max_memory_allocated() / 2**30
    if torch.cuda.is_available()
    else 0.0
)

print("V5-B SMOKE PASS")
print("loss             :", float(smoke_loss.detach().cpu()))
print("block 10 grad    :", g10)
print("block 11 grad    :", g11)
print("spatial grad     :", g_spatial)
print("peak GPU GiB     :", peak_gib)
print("frozen block9 grad: NONE")

optimizer.zero_grad(set_to_none=True)
del video, target, valid, aux, out, smoke_loss
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Train / resume


In [ ]:
resume_path = LOCAL_RUN_DIR / "latest.pt"
print("resume:", resume_path if resume_path.is_file() else None)

history = trainer.fit(
    train_loader,
    val_loader,
    epochs=tc["epochs"],
    max_train_steps=tc["max_steps_per_epoch"],
    # val_loader itself already contains exactly the uniformly sampled 800.
    max_val_steps=None,
    resume_from=resume_path if resume_path.is_file() else None,
    early_stopping_patience=tc.get("early_stopping_patience", 0),
    backfill_validation_on_resume=False,
)

history_df = pd.DataFrame([
    {
        "epoch": row["epoch"],
        "minutes": row["minutes"],
        "learning_rate": row.get("learning_rate"),
        "max_gpu_memory_gib": row.get("max_gpu_memory_gib"),
        **{f"train/{k}": v for k, v in row["train"].items()},
        **{f"val/{k}": v for k, v in row["val"].items()},
    }
    for row in history
])
display(history_df)


## Compare v5-A vs v5-B


In [ ]:
if len(history_df):
    proxy_col = "val/proxy/robust_mean_stage3_score"
    accel_col = "val/proxy/robust_mean_accel_macro_f1"

    if proxy_col in history_df.columns:
        proxy_series = pd.to_numeric(
            history_df[proxy_col], errors="coerce"
        )
        diag_idx = proxy_series.idxmax()
    else:
        diag_idx = history_df["val/total"].astype(float).idxmin()

    row = history_df.loc[diag_idx]

    summary = {
        "run_variant": RUN_VARIANT,
        "git_commit": GIT_COMMIT,
        "vjepa_commit": ACTUAL_VJEPA_COMMIT,
        "clip_len": dc["clip_len"],
        "finetune_block_indices": list(
            ft_report.trainable_block_indices
        ),
        "diagnostic_best_epoch": int(row["epoch"]),
        "diagnostic_best_proxy_stage3": (
            float(row[proxy_col])
            if proxy_col in history_df.columns
            else None
        ),
        "diagnostic_best_proxy_accel": (
            float(row[accel_col])
            if accel_col in history_df.columns
            else None
        ),
        "peak_gpu_gib": float(
            pd.to_numeric(
                history_df["max_gpu_memory_gib"],
                errors="coerce",
            ).max()
        ),
        "note": (
            "proxy thresholds are diagnostic only; official DACON "
            "labels/thresholds remain unavailable offline"
        ),
    }

    v5a_summary_path = (
        OUTPUT_ROOT
        / cfg["warm_start"]["run_name"]
        / "summary.json"
    )
    if v5a_summary_path.is_file():
        v5a_summary = json.loads(
            v5a_summary_path.read_text(encoding="utf-8")
        )
        summary["v5a_diagnostic_best_epoch"] = v5a_summary.get(
            "diagnostic_best_epoch"
        )
        summary["v5a_diagnostic_best_proxy_stage3"] = v5a_summary.get(
            "diagnostic_best_proxy_stage3"
        )
        summary["v5a_diagnostic_best_proxy_accel"] = v5a_summary.get(
            "diagnostic_best_proxy_accel"
        )

    local_summary = LOCAL_RUN_DIR / "summary.json"
    local_summary.write_text(
        json.dumps(summary, indent=2, default=str),
        encoding="utf-8",
    )
    trainer._sync_file(local_summary)
    print(json.dumps(summary, indent=2))

if WANDB_ENABLED:
    finish_wandb()

print("persistent latest:", RUN_DIR / "latest.pt")
print("persistent best  :", RUN_DIR / "best.pt")
